In [ ]:
from pathlib import Path
import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import TensorDataset, DataLoader

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import optuna
import wandb
import os
os.environ["WANDB_MODE"] = "offline"

from DataProcessing import time_gaps, make_sequences


In [ ]:
SEQ_LEN = 128
STRIDE = 1
INPUT_DIM = 51

BATCH_SIZE = 512
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
CONV_CHANNELS = 64
KERNEL_SIZE = 3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
THRESHOLD_QUANTILE = 0.99
PATIENCE = 8

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
print("device:", device)


#### Data processing

In [ ]:
def DataprocessingWithDevTest(start_time, drop_columns, SEQ_LEN, STRIDE):
    df_normal = pd.read_parquet("../Dataset/SWaT_Dataset_Normal_v1.parquet")
    df_attack = pd.read_parquet("../Dataset/SWaT_Dataset_Attack_v1.parquet")

    print("\n------------------------- Original Data -------------------------")
    print(f"Normal Data = {df_normal.shape}")
    print(f"Attack Data = {df_attack.shape}")
    print("\n------------------------- Processing ... -------------------------")

    keep_mask = df_normal["Timestamp"] >= start_time
    df_normal = df_normal.loc[keep_mask].copy()
    skipped_rows = int((~keep_mask).sum())
    print(f"Normal data = {df_normal.shape}")
    print(f"Skip data = {skipped_rows}")

    df_normal = df_normal.drop(columns=drop_columns).reset_index(drop=True)
    df_attack = df_attack.drop(columns=drop_columns).reset_index(drop=True)

    train_set, other_normal = train_test_split(df_normal, train_size=0.8, shuffle=False)
    val_set, remaining_normal = train_test_split(other_normal, train_size=0.5, shuffle=False)
    dev_normal_set, test_normal_set = train_test_split(remaining_normal, train_size=0.5, shuffle=False)
    dev_attack_set, test_attack_set = train_test_split(df_attack, train_size=0.5, shuffle=False)

    dev_set = pd.concat([dev_normal_set, dev_attack_set], axis=0, ignore_index=True)
    test_set = pd.concat([test_normal_set, test_attack_set], axis=0, ignore_index=True)

    train_gaps = time_gaps(train_set)
    val_gaps = time_gaps(val_set)
    dev_gaps = time_gaps(dev_set)
    test_gaps = time_gaps(test_set)

    def summarize_labels(name, df):
        label_counts = df["Label"].value_counts().reindex([0, 1], fill_value=0)
        print(f"{name} has {int(label_counts[0])} normal data and {int(label_counts[1])} anomaly data.")

    print(f"train_set = {train_set.shape}")
    print(f"val_set = {val_set.shape}")
    print(f"dev_set = {dev_set.shape}")
    print(f"test_set = {test_set.shape}")
    summarize_labels("dev_set", dev_set)
    summarize_labels("test_set", test_set)
    print(f"train gaps: {len(train_gaps)}")
    print(f"val gaps: {len(val_gaps)}")
    print(f"dev gaps: {len(dev_gaps)}")
    print(f"test gaps: {len(test_gaps)}")

    scaler = MinMaxScaler()
    feature_cols = train_set.columns.drop(["Timestamp", "Label"])

    train_x = scaler.fit_transform(train_set[feature_cols])
    val_x = scaler.transform(val_set[feature_cols])
    dev_x = scaler.transform(dev_set[feature_cols])
    test_x = scaler.transform(test_set[feature_cols])

    train_y = train_set["Label"].to_numpy()
    val_y = val_set["Label"].to_numpy()
    dev_y = dev_set["Label"].to_numpy()
    test_y = test_set["Label"].to_numpy()

    train_timestamps = train_set["Timestamp"].to_numpy()
    val_timestamps = val_set["Timestamp"].to_numpy()
    dev_timestamps = dev_set["Timestamp"].to_numpy()
    test_timestamps = test_set["Timestamp"].to_numpy()

    X_train_seq, y_train_seq, train_skipped = make_sequences(train_x, train_y, SEQ_LEN, STRIDE, train_timestamps)
    X_val_seq, y_val_seq, val_skipped = make_sequences(val_x, val_y, SEQ_LEN, STRIDE, val_timestamps)
    X_dev_seq, y_dev_seq, dev_skipped = make_sequences(dev_x, dev_y, SEQ_LEN, STRIDE, dev_timestamps)
    X_test_seq, y_test_seq, test_skipped = make_sequences(test_x, test_y, SEQ_LEN, STRIDE, test_timestamps)

    print("\n------------------------- Final Result -------------------------")
    print("X_train_seq:", X_train_seq.shape, "y_train_seq:", y_train_seq.shape, "skipped:", train_skipped)
    print("X_val_seq  :", X_val_seq.shape, "y_val_seq  :", y_val_seq.shape, "skipped:", val_skipped)
    print("X_dev_seq  :", X_dev_seq.shape, "y_dev_seq  :", y_dev_seq.shape, "skipped:", dev_skipped)
    print("X_test_seq :", X_test_seq.shape, "y_test_seq :", y_test_seq.shape, "skipped:", test_skipped)

    return X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_dev_seq, y_dev_seq, X_test_seq, y_test_seq


start_time = pd.to_datetime("2015-12-23 12:00:00")
drop_columns = ["Detailed_Label"]
SEQ_LEN = 128
STRIDE = 1

X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_dev_seq, y_dev_seq, X_test_seq, y_test_seq = DataprocessingWithDevTest(start_time, drop_columns, SEQ_LEN, STRIDE)


In [ ]:
# numpy -> tensor，並塑形
train_dataset = TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(np.zeros(len(X_train_seq), dtype=np.int64)))
val_dataset = TensorDataset(torch.from_numpy(X_val_seq), torch.from_numpy(np.zeros(len(X_val_seq), dtype=np.int64)))
dev_dataset = TensorDataset(torch.from_numpy(X_dev_seq), torch.from_numpy(y_dev_seq))
test_dataset = TensorDataset(torch.from_numpy(X_test_seq), torch.from_numpy(y_test_seq))


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


#### Model architecture with optuna framework

In [ ]:
def get_search_space(trial):
    params = {
        # Conv1D feature extractor
        "conv_channels": trial.suggest_categorical(
            "conv_channels", 
            [32, 64, 128, 256]
        ),

        "kernel_size": trial.suggest_categorical(
            "kernel_size", 
            [3, 5, 7]
        ),

        # LSTM encoder / decoder
        "hidden_size": trial.suggest_categorical(
            "hidden_size", 
            [32, 64, 128, 256]
        ),

        "num_layers": trial.suggest_int(
            "num_layers", 
            1, 5
        ),

        "dropout": trial.suggest_float(
            "dropout", 
            0.0, 0.5
        ),

        "bidirectional": trial.suggest_categorical(
            "bidirectional", 
            [False, True]
        ),

        # Optimizer
        "learning_rate": trial.suggest_float(
            "learning_rate", 
            1e-5, 1e-3, 
            log=True
        ),

        "weight_decay": trial.suggest_float(
            "weight_decay", 
            1e-7, 1e-3, 
            log=True
        ),
        
        "threshold_percentile": trial.suggest_float(
            "threshold_percentile",
            90.0,
            99.5
        ),
    }

    return params

In [ ]:
class ConvFeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, conv_channels: int, kernel_size: int, dropout: float):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class Encoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM，Encoder 和 Decoder 的方向性需一樣，否則維度會報錯
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x):
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # BiLSTM 的 hidden size 會翻倍 
        direction_factor = 2 if bidirectional else 1
        self.output_layer = nn.Linear(hidden_size * direction_factor, conv_channels)

    def forward(self, decoder_input, hidden, cell):
        decoded, _ = self.lstm(decoder_input, (hidden, cell))
        return self.output_layer(decoded)


class ReconstructionHead(nn.Module):
    def __init__(self, conv_channels: int, output_dim: int, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.proj = nn.Conv1d(conv_channels, output_dim, kernel_size=kernel_size, padding=padding)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.proj(x)
        return x.transpose(1, 2)


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        conv_channels: int,
        kernel_size: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool, # BiLSTM
    ):
        super().__init__()
        self.feature_extractor = ConvFeatureExtractor(input_dim, conv_channels, kernel_size, dropout)
        self.encoder = Encoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.decoder = Decoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.reconstruction_head = ReconstructionHead(conv_channels, input_dim, kernel_size)

    def forward(self, x):
        # 特徵提取
        conv_features = self.feature_extractor(x)
        # Encoder 壓縮
        hidden, cell = self.encoder(conv_features)
        # Decoder 重建
        decoder_input = torch.zeros_like(conv_features)
        decoded_features = self.decoder(decoder_input, hidden, cell)
        # 重建
        reconstruction = self.reconstruction_head(decoded_features)
        return reconstruction


In [ ]:
# model = LSTMAutoencoder(
#     input_dim=INPUT_DIM,
#     conv_channels=CONV_CHANNELS,
#     kernel_size=KERNEL_SIZE,
#     hidden_size=HIDDEN_SIZE,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     bidirectional=False, # BiLSTM
# ).to(device)

# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# print(model)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for (batch_x, _) in loader:
        batch_x = batch_x.to(device)

        optimizer.zero_grad()
        reconstruction = model(batch_x)
        loss = criterion(reconstruction, batch_x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


def evaluate_reconstruction_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for (batch_x, _) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            loss = criterion(reconstruction, batch_x)
            total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
history = {"train_loss": [], "val_loss": []}

def compute_window_score(reconstruction, batch_x, mode="mean", top_k=5):
    point_error = (reconstruction - batch_x) ** 2
    timestep_error = point_error.mean(dim=2)

    if mode == "mean":
        return timestep_error.mean(dim=1)
    if mode == "max":
        return timestep_error.max(dim=1).values
    if mode == "topk":
        k = min(top_k, timestep_error.shape[1])
        topk_vals = torch.topk(timestep_error, k=k, dim=1).values
        return topk_vals.mean(dim=1)

    raise ValueError(f"Unsupported mode: {mode}")


def get_reconstruction_errors(model, data_array: np.ndarray, device, batch_size: int = 512, score_mode: str = "mean", top_k: int = 5):
    model.eval()
    scores = []
    dataset = TensorDataset(torch.from_numpy(data_array).float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            batch_error = compute_window_score(reconstruction, batch_x, mode=score_mode, top_k=top_k)
            scores.append(batch_error.detach().cpu().numpy())

    return np.concatenate(scores)


def fit_model(params, trial=None, capture_history=False):
    model = LSTMAutoencoder(
        input_dim=INPUT_DIM,
        conv_channels=params["conv_channels"],
        kernel_size=params["kernel_size"],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"])

    local_history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state_dict = None
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate_reconstruction_loss(model, val_loader, criterion, device)

        if capture_history:
            local_history["train_loss"].append(train_loss)
            local_history["val_loss"].append(val_loss)

        print(f"Epoch [{epoch:02d}/{EPOCHS}] train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

        if trial is not None:
            trial.report(best_val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    model.load_state_dict(best_state_dict)
    return model, best_val_loss, local_history


def objective(trial):
    params = get_search_space(trial)
    score_mode = trial.suggest_categorical("score_mode", ["mean", "max", "topk"])
    top_k = trial.suggest_int("top_k", 3, 20)
    threshold_percentile = params["threshold_percentile"] / 100.0

    model, best_val_loss, _ = fit_model(params, trial=trial, capture_history=False)

    val_errors = get_reconstruction_errors(
        model,
        X_val_seq,
        device,
        batch_size=BATCH_SIZE,
        score_mode=score_mode,
        top_k=top_k,
    )
    threshold = float(np.quantile(val_errors, threshold_percentile))

    dev_errors = get_reconstruction_errors(
        model,
        X_dev_seq,
        device,
        batch_size=BATCH_SIZE,
        score_mode=score_mode,
        top_k=top_k,
    )
    dev_pred = (dev_errors > threshold).astype(int)

    dev_f1 = f1_score(y_dev_seq, dev_pred, zero_division=0)
    dev_precision = precision_score(y_dev_seq, dev_pred, zero_division=0)
    dev_recall = recall_score(y_dev_seq, dev_pred, zero_division=0)

    trial.set_user_attr("best_val_loss", best_val_loss)
    trial.set_user_attr("threshold", threshold)
    trial.set_user_attr("dev_f1", dev_f1)
    trial.set_user_attr("dev_precision", dev_precision)
    trial.set_user_attr("dev_recall", dev_recall)

    return dev_f1


In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=50)

best_params = study.best_params

print("Best dev F1:", study.best_value)
print("Best params:", best_params)
print("Best attrs:", study.best_trial.user_attrs)


In [ ]:
best_model, best_val_loss, history = fit_model(best_params, capture_history=True)

best_score_mode = best_params["score_mode"]
best_top_k = best_params["top_k"]
best_threshold_percentile = best_params["threshold_percentile"] / 100.0

val_errors = get_reconstruction_errors(
    best_model,
    X_val_seq,
    device,
    batch_size=BATCH_SIZE,
    score_mode=best_score_mode,
    top_k=best_top_k,
)
dev_errors = get_reconstruction_errors(
    best_model,
    X_dev_seq,
    device,
    batch_size=BATCH_SIZE,
    score_mode=best_score_mode,
    top_k=best_top_k,
)
test_errors = get_reconstruction_errors(
    best_model,
    X_test_seq,
    device,
    batch_size=BATCH_SIZE,
    score_mode=best_score_mode,
    top_k=best_top_k,
)

threshold = float(np.quantile(val_errors, best_threshold_percentile))
dev_pred = (dev_errors > threshold).astype(int)
test_pred = (test_errors > threshold).astype(int)

def summarize_metrics(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"[{name}]")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print()

summarize_metrics("Dev", y_dev_seq, dev_pred)
summarize_metrics("Final Test", y_test_seq, test_pred)
print(f"Threshold: {threshold:.6f}")


#### Experimental Result

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss value")
plt.title("Training Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
print(f"Best score_mode: {best_score_mode}")
print(f"Best top_k: {best_top_k}")
print(f"Best threshold_percentile: {best_threshold_percentile:.4f}")


In [ ]:
print("val_errors shape:", val_errors.shape)
print("dev_errors shape:", dev_errors.shape)
print("test_errors shape:", test_errors.shape)
print("threshold_percentile:", best_threshold_percentile)
print("threshold:", threshold)


In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(val_errors, bins=60, alpha=0.7, label="val_normal_errors")
plt.axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.6f}")
plt.xlabel("Reconstruction Error")
plt.ylabel("Count")
plt.title("Validation Error Distribution")
plt.legend()
plt.show()


In [ ]:
y_pred = test_pred

acc = accuracy_score(y_test_seq, y_pred)
precision = precision_score(y_test_seq, y_pred, zero_division=0)
recall = recall_score(y_test_seq, y_pred, zero_division=0)
f1 = f1_score(y_test_seq, y_pred, zero_division=0)
cm = confusion_matrix(y_test_seq, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nClassification Report")
print(classification_report(y_test_seq, y_pred, digits=4, zero_division=0))
print("Confusion Matrix:\n", cm)


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test_errors, label="reconstruction_error", linewidth=1)
plt.plot(y_test_seq * threshold, label="true_anomaly(label scaled)", alpha=0.8)
plt.axhline(threshold, color="red", linestyle="--", label="threshold")
plt.xlabel("Sequence Index")
plt.ylabel("Error")
plt.title("Final Test Reconstruction Error")
plt.legend()
plt.show()
